# Ex 7

Un cercetător măsoară nivelul de zgomot ambiental în diferite locaţii dintr-un oraş. El suspectează că nivelul de
zgomot, notat X, urmează o distribuţie normală, dar atât media μ, cât şi deviaţia standard σ sunt necunoscute.
Cercetătorul are 10 observaţii ale nivelului de zgomot (în decibeli, dB):
56, 60, 58, 55, 57, 59, 61, 56, 58, 60.
Scopul este să folosim inferenţa Bayesiană pentru a estima pe μ şi pe σ.

a) Definiţi un model în PyMC pentru această problemă. Puteţi alege ca distribuţii a priori μ ∼ N (x, 102) şi σ ∼HalfNormal(10). Cât ar trebui să fie x?

b) Inferaţi asupra lui μ şi σ. Care este intervalul de încredere (HDI 95%) pentru μ şi σ?

Suma observațiilor este $56 + 60 + 58 + 55 + 57 + 59 + 61 + 56 + 58 + 60 = 580$. Împărțind la numărul de observații (10), obținem media $580 / 10 = 58$. Prin urmare, valoarea lui $x$ ar trebui să fie 58. Aceasta centrează distribuția a priori pe media datelor observate, o practică frecventă pentru priorii slab informativi.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np

zgomot = np.array([56, 60, 58, 55, 57, 59, 61, 56, 58, 60])

# a) 
x = zgomot.mean() # 58.0

with pm.Model() as model_zgomot: 
    mu = pm.Normal("mu", mu=x, sigma=10) # 10^2 este varianța, deci sigma=10
    sigma = pm.HalfNormal("sigma", sigma=10)
    
    # Mecanismul de generare a datelor (Likelihood)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=zgomot)
    
    # b) 
    # Motorul de inferență explorează posteriorul și produce eșantioane
    idata = pm.sample(draws=1000, tune=1000, chains=4, random_seed=42, progressbar=False) 
    
    # Extragerea intervalului HDI 95%
    # Parametrul ci_prob=0.95 modifică default-ul de 94% al ArviZ
    rezumat = az.summary(idata, var_names=["mu", "sigma"], ci_prob=0.95) 
    print(rezumat)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu, sigma]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 14 seconds.


        mean    sd eti95_lb eti95_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
mu     57.99  0.76       56       60     2302     2255  1.00     0.016   0.017
sigma   2.34  0.72      1.4      4.2     2239     2019  1.00     0.016   0.024
